## Import

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve_triangular
import scipy.linalg as la

import sys
print(sys.version)

from sksparse.cholmod import cholesky as cholmod_cholesky
print("scikit-sparse importato correttamente")

3.11.16 | packaged by conda-forge | (main, Aug 21 2026, 22:37:11) [MSC v.1944 64 bit (AMD64)]
scikit-sparse importato correttamente


## Carica dati

In [7]:
# Caricamento del vettore rhs.txt
rhs = np.loadtxt("../FileGenerati/rhs.txt")
num_incognite = len(rhs)
print(f"Dimensioni del vettore rhs: {num_incognite}")

# Caricamento della matrice sparsa A.txt
dati_A = np.loadtxt("../FileGenerati/A.txt")
righe = dati_A[:, 0].astype(int)
colonne = dati_A[:, 1].astype(int)
valori = dati_A[:, 2]
print(f"Dimensioni della matrice A: {dati_A.shape}")

Dimensioni del vettore rhs: 1024
Dimensioni della matrice A: (4992, 3)


## COO CSC e Cholesky

In [ ]:
# Costruzione della matrice sparsa A in formato COO
A_coo = sp.coo_matrix((valori, (righe, colonne)), shape=(num_incognite, num_incognite))

# Conversione della matrice A in formato CSC (Compressed Sparse Column)
A_csc = A_coo.tocsc()
print(f"Matrice A convertita con successo in formato CSC ({A_csc.shape[0]}x{A_csc.shape[1]}).")


# 2. Fattorizzazione di Cholesky: -A = L * L^T
print("\n--- 2. Fattorizzazione di Cholesky ---")
# Definiamo la funzione my_cholesky come indicato nelle specifiche
def my_cholesky(A):
    # -A è definita positiva (A è la matrice del sistema, negativa definita)
    # il segno meno su una matrice sparsa NON la densifica: resta sparsa
    neg_A = -A

    # order=None (NON la stringa "natural"!): in questa versione di scikit-sparse,
    # order=None restituisce DIRETTAMENTE la matrice, senza incapsularla in una
    # tupla con permutazione - e non applica nessun riordinamento interno,
    # rispettando l'ordinamento che le passiamo noi (naturale o nested-dissection
    # dal Task 2/3). lower=True è necessario per ottenere L (triangolare
    # inferiore) invece di R (triangolare superiore, il default).
    L = cholmod_cholesky(neg_A, order=None, lower=True)

    return sp.csc_matrix(L)

# Eseguiamo la fattorizzazione di Cholesky per ottenere la matrice L
L = my_cholesky(A_csc)
print("Fattorizzazione di Cholesky -A = L * L^T eseguita con successo.")
print(f"Dimensione del fattore L: {L.shape}")


Matrice A convertita con successo in formato CSC (1024x1024).

--- 2. Fattorizzazione di Cholesky ---
Fattorizzazione di Cholesky -A = L * L^T eseguita con successo.
Dimensione del fattore L: (1024, 1024)


## Risoluzione sistema lineare

-A * u = -rhs  =>  L * L^T * u = -rhs

In [9]:

# Nota: Dato che fattorizziamo -A, trasformiamo l'equazione A * u = rhs in (-A) * u = -rhs
b = -rhs

# Passaggio 3a: Risoluzione del sistema triangolare inferiore L * y = b
y = spsolve_triangular(L, b, lower=True)

# Passaggio 3b: Risoluzione del sistema triangolare superiore L^T * u = y
L_T = L.transpose().tocsc()
u = spsolve_triangular(L_T, y, lower=False)

print("Sistema risolto con successo!\n")


Sistema risolto con successo!



## Verifica soluzione e termine residuo

In [10]:
# 4. Verifica della soluzione ed Errore di Residuo

residuo = np.linalg.norm(A_csc.dot(u) - rhs)
print(f"Norma del residuo ||A*u - rhs||: {residuo:.2e}")

print("\nPrime componenti della soluzione approssimata u:")
for idx, val in enumerate(u[:min(9, len(u))]):
    print(f"  u[{idx}] = {val:.6f}")

Norma del residuo ||A*u - rhs||: 7.65e-14

Prime componenti della soluzione approssimata u:
  u[0] = 0.110693
  u[1] = 0.176307
  u[2] = 0.215122
  u[3] = 0.235963
  u[4] = 0.243944
  u[5] = 0.242503
  u[6] = 0.234202
  u[7] = 0.221071
  u[8] = 0.204761
